# PyTorch desde tensores hasta una CNN


## Resultados de aprendizaje

Al terminar podrás:

1. crear y transformar tensores;
2. calcular gradientes con `autograd`;
3. preparar datos con `TensorDataset` y `DataLoader`;
4. construir, entrenar y evaluar una MLP;
5. entender canales, convolución y *pooling*;
6. construir una CNN, compararla con la MLP y guardar sus pesos.

> **Entorno recomendado:** Google Colab o Python 3.10+ con PyTorch, scikit-learn y Matplotlib. Si te falta alguna dependencia, ejecuta en una celda aparte:  
> `%pip install torch scikit-learn matplotlib`

No se necesita descargar el dataset: `load_digits()` viene incluido en scikit-learn.


## Bloque A — Fundamentos de PyTorch


## Ejercicio 1 — Entorno, semilla y dispositivo

**Objetivo:** preparar un experimento reproducible y elegir automáticamente CPU, CUDA o MPS.

1. Importa las bibliotecas indicadas.
2. Define `SEED = 42` y fija la semilla de Python, NumPy y PyTorch.
3. Guarda el dispositivo disponible en `DEVICE`.

**Comprobación:** imprime la versión de PyTorch y el dispositivo elegido.


In [ ]:
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("PyTorch:", torch.__version__)
print("Dispositivo:", DEVICE)


**Idea clave:** La semilla reduce variaciones entre ejecuciones. Aun así, algunos algoritmos de GPU pueden producir diferencias pequeñas.


## Ejercicio 2 — Crear e inspeccionar tensores

**Objetivo:** practicar `shape`, `dtype`, operaciones elemento a elemento y reducciones.

Crea un tensor flotante `a` de forma `(2, 3)` con los valores del 1 al 6. Después:

- crea `b` con unos y la misma forma;
- calcula `a + b` y `a * b`;
- calcula la media de cada fila;
- imprime forma, tipo y dispositivo.


In [ ]:
a = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)
b = torch.ones_like(a)

suma = a + b
producto = a * b
media_por_fila = a.mean(dim=1)

print("a =\n", a)
print("shape:", a.shape, "dtype:", a.dtype, "device:", a.device)
print("a + b =\n", suma)
print("a * b =\n", producto)
print("Media por fila:", media_por_fila)

assert a.shape == (2, 3)
assert torch.allclose(media_por_fila, torch.tensor([2.0, 5.0]))


**Idea clave:** En PyTorch, `dim` indica el eje que se reduce. `mean(dim=1)` resume las columnas y entrega una media por fila.


## Ejercicio 3 — Cambiar formas y vectorizar

**Objetivo:** pasar del formato de imágenes `(N, C, H, W)` al formato tabular `(N, características)` que usa una MLP.

1. Crea un lote ficticio de 2 imágenes, 1 canal y 8×8 píxeles.
2. Aplánalo conservando la dimensión del lote.
3. Multiplica el resultado por una matriz de pesos de forma `(64, 10)`.

**Resultado esperado:** los *logits* tendrán forma `(2, 10)`.


In [ ]:
images_demo = torch.arange(2 * 1 * 8 * 8, dtype=torch.float32).reshape(2, 1, 8, 8)
flat_demo = images_demo.flatten(start_dim=1)
weights_demo = torch.randn(64, 10)
logits_demo = flat_demo @ weights_demo

print("Imágenes:", images_demo.shape)
print("Aplanadas:", flat_demo.shape)
print("Logits:", logits_demo.shape)

assert flat_demo.shape == (2, 64)
assert logits_demo.shape == (2, 10)


**Idea clave:** La primera dimensión es el lote y debe conservarse. Cada una de las 10 salidas será una puntuación para una clase.


## Ejercicio 4 — Gradientes automáticos con `autograd`

**Objetivo:** observar el mecanismo que usa PyTorch durante el aprendizaje.

Para el modelo $\hat{y}=wx+b$, usa `x=3`, `y=10`, `w=2` y `b=1`.

1. Activa el seguimiento de gradientes en `w` y `b`.
2. Calcula el error cuadrático $(\hat{y}-y)^2$.
3. Ejecuta `backward()` e imprime `w.grad` y `b.grad`.
4. Antes de ejecutar, intenta anticipar ambos gradientes.


In [ ]:
x = torch.tensor(3.0)
y_true = torch.tensor(10.0)
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

y_pred = w * x + b
loss = (y_pred - y_true) ** 2
loss.backward()

print("Predicción:", y_pred.item())
print("Pérdida:", loss.item())
print("dL/dw:", w.grad.item())
print("dL/db:", b.grad.item())

assert w.grad.item() == -18.0
assert b.grad.item() == -6.0


**Idea clave:** `backward()` recorre el grafo de operaciones con la regla de la cadena. Los gradientes indican cómo cambiaría la pérdida ante un cambio pequeño de cada parámetro.


## Bloque B — Datos y lotes


## Ejercicio 5 — Cargar y explorar imágenes

**Objetivo:** conocer la entrada antes de modelarla.

1. Carga `load_digits()`.
2. Guarda las imágenes como `float32` en `X` y las etiquetas como `int64` en `y`.
3. Imprime formas, rango de píxeles y cantidad de clases.
4. Muestra un ejemplo de cada dígito del 0 al 9.

**Pregunta breve:** ¿por qué aún no conviene pasar directamente estos valores a la red?


In [ ]:
digits = load_digits()
X = digits.images.astype(np.float32)
y = digits.target.astype(np.int64)

print("X:", X.shape, "y:", y.shape)
print("Rango de píxeles:", X.min(), X.max())
print("Clases:", np.unique(y))

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for digit, ax in enumerate(axes.ravel()):
    idx = np.flatnonzero(y == digit)[0]
    ax.imshow(X[idx], cmap="gray_r")
    ax.set_title(f"Clase {digit}")
    ax.axis("off")
plt.suptitle("Un ejemplo por clase", y=1.02)
plt.tight_layout()
plt.show()


**Idea clave:** Los píxeles van de 0 a 16. Los normalizaremos a `[0, 1]` para que la optimización sea más estable.


## Ejercicio 6 — Dividir datos y crear `DataLoader`

**Objetivo:** construir un flujo de entrenamiento correcto.

1. Divide los datos en 70% entrenamiento, 15% validación y 15% prueba, conservando la proporción de clases (`stratify`).
2. Convierte las imágenes a tensores `(N, 1, 8, 8)` y normaliza dividiendo entre 16.
3. Usa `TensorDataset` y crea tres `DataLoader` con lote de 64.
4. Baraja únicamente entrenamiento.

**Comprobación:** un lote de imágenes debe medir `(64, 1, 8, 8)` y sus etiquetas `(64,)`.


In [ ]:
X_train_np, X_temp_np, y_train_np, y_temp_np = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val_np, X_test_np, y_val_np, y_test_np = train_test_split(
    X_temp_np, y_temp_np, test_size=0.50, random_state=SEED, stratify=y_temp_np
)

def to_tensors(images, labels):
    image_tensor = torch.from_numpy(images).unsqueeze(1).float() / 16.0
    label_tensor = torch.from_numpy(labels).long()
    return image_tensor, label_tensor

X_train, y_train = to_tensors(X_train_np, y_train_np)
X_val, y_val = to_tensors(X_val_np, y_val_np)
X_test, y_test = to_tensors(X_test_np, y_test_np)

train_ds = TensorDataset(X_train, y_train)
val_ds = TensorDataset(X_val, y_val)
test_ds = TensorDataset(X_test, y_test)

BATCH_SIZE = 64
generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, generator=generator
)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

xb, yb = next(iter(train_loader))
print("Particiones:", len(train_ds), len(val_ds), len(test_ds))
print("Lote:", xb.shape, yb.shape, xb.dtype, yb.dtype)
print("Rango normalizado:", xb.min().item(), xb.max().item())

assert xb.ndim == 4 and xb.shape[1:] == (1, 8, 8)
assert yb.dtype == torch.int64


**Idea clave:** Validación sirve para tomar decisiones durante el desarrollo; prueba se reserva para la evaluación final. `DataLoader` evita cargar y procesar todo como un único lote.


## Bloque C — Perceptrón multicapa (MLP)


## Ejercicio 7 — Primera capa `Linear`

**Objetivo:** entender una capa totalmente conectada.

1. Aplana un lote real a `(N, 64)`.
2. Crea `nn.Linear(64, 10)` y pásala al dispositivo.
3. Realiza un `forward` y cuenta sus parámetros entrenables.
4. Verifica manualmente que hay $64\times10+10=650$ parámetros.


In [ ]:
flat_xb = xb.flatten(start_dim=1).to(DEVICE)
linear = nn.Linear(64, 10).to(DEVICE)
linear_logits = linear(flat_xb)
n_params_linear = sum(p.numel() for p in linear.parameters() if p.requires_grad)

print("Entrada:", flat_xb.shape)
print("Salida:", linear_logits.shape)
print("Parámetros:", n_params_linear)

assert linear_logits.shape == (xb.shape[0], 10)
assert n_params_linear == 64 * 10 + 10


**Idea clave:** Cada neurona de salida tiene 64 pesos y un sesgo. La capa entrega puntuaciones, no probabilidades.


## Ejercicio 8 — Construir una MLP

**Objetivo:** implementar una red con capas ocultas.

Crea `MLP`, heredando de `nn.Module`, con esta arquitectura:

`Flatten → Linear(64,64) → ReLU → Dropout(0.2) → Linear(64,32) → ReLU → Linear(32,10)`

Implementa `forward`, instancia el modelo y verifica que un lote produce `(N, 10)`.


In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 10),
        )

    def forward(self, x):
        return self.net(x)

mlp = MLP().to(DEVICE)
mlp_logits = mlp(xb.to(DEVICE))

print(mlp)
print("Salida:", mlp_logits.shape)
assert mlp_logits.shape == (xb.shape[0], 10)


**Idea clave:** `ReLU` introduce no linealidad. `Dropout` apaga activaciones al azar solo durante entrenamiento y ayuda a regularizar.


## Ejercicio 9 — Logits, probabilidades y entropía cruzada

**Objetivo:** conectar la salida del modelo con una tarea multiclase.

1. Convierte los *logits* en probabilidades con `softmax(dim=1)`.
2. Verifica que cada fila suma aproximadamente 1.
3. Calcula `CrossEntropyLoss` usando directamente los *logits* y las etiquetas.

**Pregunta clave:** ¿por qué no se pasa `softmax(logits)` a `CrossEntropyLoss`?


In [ ]:
probabilities = torch.softmax(mlp_logits, dim=1)
loss_fn = nn.CrossEntropyLoss()
classification_loss = loss_fn(mlp_logits, yb.to(DEVICE))

print("Suma de las primeras probabilidades:", probabilities[0].sum().item())
print("Pérdida:", classification_loss.item())

assert torch.allclose(
    probabilities.sum(dim=1),
    torch.ones(probabilities.shape[0], device=DEVICE),
    atol=1e-6,
)


**Idea clave:** `CrossEntropyLoss` combina internamente `LogSoftmax` y `NLLLoss`, con una formulación numéricamente estable. Por eso recibe logits crudos.


## Ejercicio 10 — Ejecutar un paso de optimización

**Objetivo:** completar el ciclo mínimo de aprendizaje.

Con `Adam(lr=1e-3)`, ejecuta sobre un lote:

1. `zero_grad()`;
2. `forward`;
3. cálculo de pérdida;
4. `backward()`;
5. `step()`.

Imprime la pérdida y la norma del gradiente de la primera capa lineal.


In [ ]:
set_seed()
demo_mlp = MLP().to(DEVICE)
optimizer = torch.optim.Adam(demo_mlp.parameters(), lr=1e-3)

demo_mlp.train()
xb_device, yb_device = xb.to(DEVICE), yb.to(DEVICE)

optimizer.zero_grad(set_to_none=True)
logits = demo_mlp(xb_device)
batch_loss = loss_fn(logits, yb_device)
batch_loss.backward()
first_grad_norm = demo_mlp.net[1].weight.grad.norm().item()
optimizer.step()

print("Pérdida del lote:", batch_loss.item())
print("Norma del gradiente:", first_grad_norm)


**Idea clave:** Los gradientes se acumulan por defecto; por eso deben limpiarse antes de cada nuevo lote.


## Ejercicio 11 — Crear ciclos reutilizables de entrenamiento y evaluación

**Objetivo:** evitar duplicación y aplicar correctamente `train()`, `eval()` y `no_grad()`.

Completa:

- `run_epoch(...)`: entrena si recibe optimizador; en otro caso solo evalúa;
- acumulación de pérdida media y exactitud;
- `fit(...)`: ejecuta varias épocas y guarda el historial de entrenamiento/validación.


In [ ]:
def run_epoch(model, loader, loss_fn, optimizer=None):
    is_training = optimizer is not None

    if is_training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    context = torch.enable_grad() if is_training else torch.no_grad()
    with context:
        for inputs, targets in loader:
            inputs = inputs.to(DEVICE)
            targets = targets.to(DEVICE)

            if is_training:
                optimizer.zero_grad()

            logits = model(inputs)
            loss = loss_fn(logits, targets)

            if is_training:
                loss.backward()
                optimizer.step()

            batch_size = targets.size(0)
            total_loss += loss.item() * batch_size
            total_correct += (logits.argmax(dim=1) == targets).sum().item()
            total_examples += batch_size

    return {
        "loss": total_loss / total_examples,
        "accuracy": total_correct / total_examples,
    }


def fit(model, train_loader, val_loader, loss_fn, optimizer, epochs=20):
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(1, epochs + 1):
        train_metrics = run_epoch(model, train_loader, loss_fn, optimizer)
        val_metrics = run_epoch(model, val_loader, loss_fn)

        history["train_loss"].append(train_metrics["loss"])
        history["val_loss"].append(val_metrics["loss"])
        history["train_acc"].append(train_metrics["accuracy"])
        history["val_acc"].append(val_metrics["accuracy"])

        if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
            print(
                f"Época {epoch:02d}/{epochs} | "
                f"train loss={train_metrics['loss']:.4f}, acc={train_metrics['accuracy']:.3f} | "
                f"val loss={val_metrics['loss']:.4f}, acc={val_metrics['accuracy']:.3f}"
            )

    return history


**Idea clave:** `model.eval()` cambia el comportamiento de capas como Dropout. `torch.no_grad()` evita construir el grafo y ahorra memoria durante evaluación.


## Ejercicio 12 — Entrenar la MLP y leer sus curvas

**Objetivo:** entrenar el primer modelo completo y diagnosticar su evolución.

1. Reinicia la semilla e instancia una MLP nueva.
2. Entrénala 20 épocas con Adam y `lr=1e-3`.
3. Grafica pérdida y exactitud para entrenamiento y validación.
4. Describe si observas subajuste, ajuste razonable o sobreajuste.


In [ ]:
def plot_history(history, title):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    axes[0].plot(epochs, history["train_loss"], label="Entrenamiento")
    axes[0].plot(epochs, history["val_loss"], label="Validación")
    axes[0].set(title=f"{title}: pérdida", xlabel="Época", ylabel="Cross-entropy")
    axes[0].legend()

    axes[1].plot(epochs, history["train_acc"], label="Entrenamiento")
    axes[1].plot(epochs, history["val_acc"], label="Validación")
    axes[1].set(title=f"{title}: exactitud", xlabel="Época", ylabel="Accuracy", ylim=(0, 1.02))
    axes[1].legend()

    plt.tight_layout()
    plt.show()

set_seed()
mlp = MLP().to(DEVICE)
loss_fn = nn.CrossEntropyLoss()
mlp_optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3)
history_mlp = fit(
    mlp, train_loader, val_loader, loss_fn, mlp_optimizer, epochs=20
)

plot_history(history_mlp, "MLP")


**Idea clave:** Una brecha creciente —entrenamiento mejora mientras validación empeora— sugiere sobreajuste. No juzgues usando el conjunto de prueba durante el desarrollo.


## Ejercicio 13 — Evaluar la MLP

**Objetivo:** medir desempeño más allá de una sola cifra.

1. Evalúa pérdida y exactitud en prueba.
2. Implementa `collect_predictions` sin gradientes.
3. Dibuja la matriz de confusión.
4. Revisa precisión, *recall* y F1 por clase con `classification_report`.

**Pregunta breve:** ¿qué pares de dígitos confunde más la MLP?


In [ ]:
def collect_predictions(model, loader):
    model.eval()
    all_targets = []
    all_predictions = []

    with torch.no_grad():
        for inputs, targets in loader:
            logits = model(inputs.to(DEVICE))
            predictions = logits.argmax(dim=1).cpu()
            all_targets.append(targets.cpu())
            all_predictions.append(predictions)

    return (
        torch.cat(all_targets).numpy(),
        torch.cat(all_predictions).numpy(),
    )

test_metrics_mlp = run_epoch(mlp, test_loader, loss_fn)
y_true_mlp, y_pred_mlp = collect_predictions(mlp, test_loader)

print("Métricas MLP:", test_metrics_mlp)
ConfusionMatrixDisplay.from_predictions(
    y_true_mlp, y_pred_mlp, cmap="Blues", colorbar=False
)
plt.title("Matriz de confusión — MLP")
plt.show()

print(classification_report(y_true_mlp, y_pred_mlp, digits=3, zero_division=0))


**Idea clave:** La diagonal representa aciertos. Las celdas fuera de la diagonal revelan qué clases se confunden entre sí.


## Bloque D — Redes convolucionales (CNN)


## Ejercicio 14 — Convolución, canales y *pooling*

**Objetivo:** observar cómo cambia la forma de un lote dentro de una CNN.

Sobre cuatro imágenes `(4, 1, 8, 8)`:

1. aplica `Conv2d(1, 8, kernel_size=3, padding=1)` y ReLU;
2. aplica `MaxPool2d(2)`;
3. imprime las tres formas.

Explica por qué la convolución conserva 8×8 y el *pooling* produce 4×4.


In [ ]:
sample_images = xb[:4].to(DEVICE)
conv = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1).to(DEVICE)
pool = nn.MaxPool2d(kernel_size=2)

with torch.no_grad():
    feature_maps = torch.relu(conv(sample_images))
    pooled_maps = pool(feature_maps)

print("Entrada:", sample_images.shape)
print("Tras Conv2d:", feature_maps.shape)
print("Tras MaxPool2d:", pooled_maps.shape)

assert feature_maps.shape == (4, 8, 8, 8)
assert pooled_maps.shape == (4, 8, 4, 4)


**Idea clave:** El formato es `(lote, canales, alto, ancho)`. Con kernel 3, stride 1 y padding 1, el tamaño espacial se conserva; pooling 2×2 lo reduce a la mitad.


## Ejercicio 15 — Construir una CNN pequeña

**Objetivo:** combinar extracción espacial de características y clasificación.

Implementa:

`Conv(1→16, 3, pad=1) → ReLU → Pool(2) → Conv(16→32, 3, pad=1) → ReLU → Pool(2) → Flatten → Linear(128→64) → ReLU → Dropout(0.2) → Linear(64→10)`

Justifica por qué, después del segundo *pooling*, quedan `32×2×2 = 128` características.


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 2 * 2, 64),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

cnn = SmallCNN().to(DEVICE)
cnn_logits = cnn(xb.to(DEVICE))

print(cnn)
print("Salida:", cnn_logits.shape)
assert cnn_logits.shape == (xb.shape[0], 10)


**Idea clave:** A diferencia de la MLP, la CNN conserva vecindades espaciales y reutiliza el mismo filtro en toda la imagen.


## Ejercicio 16 — Entrenar y evaluar la CNN

**Objetivo:** aplicar el mismo ciclo de entrenamiento a otra arquitectura.

1. Reinicia la semilla e instancia una CNN nueva.
2. Entrénala 20 épocas con Adam y `lr=1e-3`.
3. Grafica las curvas.
4. Evalúa el conjunto de prueba y crea su matriz de confusión.


In [ ]:
set_seed()
cnn = SmallCNN().to(DEVICE)
cnn_optimizer = torch.optim.Adam(cnn.parameters(), lr=1e-3)
history_cnn = fit(
    cnn, train_loader, val_loader, loss_fn, cnn_optimizer, epochs=20
)

plot_history(history_cnn, "CNN")

test_metrics_cnn = run_epoch(cnn, test_loader, loss_fn)
y_true_cnn, y_pred_cnn = collect_predictions(cnn, test_loader)

print("Métricas CNN:", test_metrics_cnn)
ConfusionMatrixDisplay.from_predictions(
    y_true_cnn, y_pred_cnn, cmap="Greens", colorbar=False
)
plt.title("Matriz de confusión — CNN")
plt.show()


**Idea clave:** La infraestructura de datos y entrenamiento no depende de la arquitectura; basta con que el modelo reciba el lote y entregue logits por clase.


## Bloque E — Comparación, inferencia y persistencia


## Ejercicio 17 — Comparar MLP y CNN

**Objetivo:** relacionar complejidad del modelo y generalización.

1. Implementa `count_parameters(model)`.
2. Compara número de parámetros, pérdida de prueba y exactitud de MLP/CNN.
3. Responde:
   - ¿cuál generaliza mejor en esta ejecución?;
   - ¿más parámetros implicaron necesariamente más exactitud?;
   - ¿qué ventaja inductiva tiene la CNN para imágenes?


In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

comparison = {
    "MLP": {
        "parameters": count_parameters(mlp),
        "test_loss": test_metrics_mlp["loss"],
        "test_accuracy": test_metrics_mlp["accuracy"],
    },
    "CNN": {
        "parameters": count_parameters(cnn),
        "test_loss": test_metrics_cnn["loss"],
        "test_accuracy": test_metrics_cnn["accuracy"],
    },
}

for model_name, metrics in comparison.items():
    print(
        f"{model_name:>3} | parámetros={metrics['parameters']:,} | "
        f"test loss={metrics['test_loss']:.4f} | "
        f"test acc={metrics['test_accuracy']:.3f}"
    )


**Idea clave:** La comparación es empírica: puede variar ligeramente. La CNN incorpora el supuesto útil de que los patrones cercanos y trasladados dentro de una imagen importan.


## Ejercicio 18 — Guardar, recargar y analizar errores

**Objetivo:** cerrar el flujo de trabajo de un modelo.

1. Guarda `cnn.state_dict()` en `cnn_digits_state_dict.pt`.
2. Crea una CNN nueva, carga los pesos con `map_location=DEVICE` y activa `eval()`.
3. Verifica que sus predicciones coincidan con las del modelo original.
4. Visualiza hasta 10 errores con etiqueta real y predicción.

**Reto opcional:** elige un error y razona qué característica visual pudo confundir al modelo.


In [ ]:
checkpoint_path = Path("cnn_digits_state_dict.pt")
torch.save(cnn.state_dict(), checkpoint_path)

loaded_cnn = SmallCNN().to(DEVICE)
try:
    state_dict = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
except TypeError:  # Compatibilidad con versiones antiguas de PyTorch
    state_dict = torch.load(checkpoint_path, map_location=DEVICE)
loaded_cnn.load_state_dict(state_dict)
loaded_cnn.eval()

y_true_loaded, y_pred_loaded = collect_predictions(loaded_cnn, test_loader)
assert np.array_equal(y_true_loaded, y_true_cnn)
assert np.array_equal(y_pred_loaded, y_pred_cnn)
print("Predicciones idénticas tras recargar:", True)
print("Checkpoint guardado en:", checkpoint_path.resolve())

wrong_indices = np.flatnonzero(y_pred_cnn != y_true_cnn)
indices_to_show = wrong_indices[:10]

if len(indices_to_show) == 0:
    print("La CNN no cometió errores en este conjunto de prueba.")
else:
    fig, axes = plt.subplots(2, 5, figsize=(10, 4))
    for ax in axes.ravel():
        ax.axis("off")
    for ax, idx in zip(axes.ravel(), indices_to_show):
        ax.imshow(X_test[idx, 0].numpy(), cmap="gray_r")
        ax.set_title(f"Real {y_true_cnn[idx]} | Pred {y_pred_cnn[idx]}", color="crimson")
        ax.axis("off")
    plt.suptitle("Errores de la CNN", y=1.02)
    plt.tight_layout()
    plt.show()


**Idea clave:** `state_dict` contiene parámetros aprendidos, no la definición de la clase. Para recargarlo debes reconstruir la misma arquitectura.


## Cierre

Has recorrido el flujo esencial de PyTorch:

`tensores → gradientes → DataLoader → nn.Module → pérdida/optimizador → entrenamiento → evaluación → MLP → CNN → guardado`

### Próximos pasos sugeridos

- usar `FashionMNIST` o `CIFAR-10` con `torchvision`;
- añadir transformaciones de datos (*data augmentation*);
- probar *early stopping*, regularización y ajuste de hiperparámetros;
- mover el mismo código a una GPU y medir el tiempo de entrenamiento.
